# Hybrid MPPT Notebook Documentation

## Step 1: Notebook Overview

This notebook documents a hybrid maximum power point tracking (MPPT) workflow for photovoltaic (PV) curves under partial shading. The process keeps the original algorithm intact: load PV data, clean and validate I-V/P-V curves, collect a sparse 12-point scan, train a Physics-Informed TCNformer to predict an MPP seed/zone, and refine that seed with local Perturb-and-Observe (P&O).

**Why this matters:** under partial shading, the P-V curve can have multiple peaks, so a controller must avoid settling on a local maximum. The hybrid approach uses sparse measurements to estimate a promising region and then uses local refinement to improve the final operating point.

**What to check:** confirm that each later section runs in order and that the final metrics/plots compare the true MPP, neural seed, refined MPP, and coarse 12-point baseline.


## Step 2: Import Libraries and Setup

This step imports numerical, plotting, machine-learning, and dataset-loading libraries. It also keeps the original top-level notebook description so the execution context is clear.

**Why this matters for partial-shading MPPT:** PV curves must be processed as arrays, summarized as metrics, plotted for review, and passed into a neural model.

**What to check:** the code should print no import errors. If SciPy is unavailable, `.mat` loading will not work, matching the original fallback behavior.


In [ ]:
"""Hybrid PV MPPT pipeline using sparse sampling, learning-based seeding, and local refinement.

Pipeline overview
-----------------
1) Load PV curves from .mat/.npz sources (supports both I-V and direct P-V style encodings).
2) Extract voltage/current-like vectors from heterogeneous MATLAB/Python container layouts.
3) Clean curves with PV endpoint constraints (I(0)≈Isc, I(Voc)=0) and validate physical consistency.
4) Build 12-point sparse scan features that emulate practical MPPT measurement budgets.
5) Train/use a Physics-Informed TCNformer to predict an initial MPP voltage zone + offset.
6) Refine the seed with local Perturb-and-Observe (P&O) to recover final tracking accuracy.
7) Report seed-only (TCNformer) and refined (TCNformer + local P&O) metrics and visualizations.
"""

# Standard-library utilities for file checks, reproducibility, and typed data containers.
import os
import random
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Sequence, Tuple

# Scientific Python stack used for PV curve arrays, tabular metrics, and plots.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PyTorch implements the TCN/Transformer/ANN model and training loop.
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

try:
    from scipy.io import loadmat
except Exception:
    loadmat = None

# =========================================================


## Step 3: User Configuration

This step defines dataset selection flags, MATLAB key names, and the `Config` values used throughout training, evaluation, P&O refinement, and plotting.

**Why this matters for partial-shading MPPT:** these settings define the 12-point scan range, zone boundaries, model size, loss weights, and local refinement behavior. They are intentionally not changed in this documentation update.

**What to check:** verify that the dataset path/key configuration matches the file being used and that the printed configuration reflects the expected device and limits.


In [ ]:
# USER CONFIG
# Dataset flags: path selection and run-time outputs (plots/model bundle).
# MAT key groups map explicit simulated/shaded/normal experimental sources.
# =========================================================
DATASET_PATH = None
MAKE_PLOTS = True
SAVE_MODEL_BUNDLE = True
REPORT_OK_EXPERIMENTAL_AUX = True
NEW_DATASET_FORMAT = "vp"

# Explicit MAT selection for the Zenodo PV dataset
MAT_SIM_KEYS = ["full_curvesOk_simulated", "full_curvesSh_simulated"]
MAT_EXP_SH_KEYS = ["full_curvesSh_experimental"]
MAT_EXP_OK_KEYS = ["full_curvesOk_experimental"]

@dataclass
class Config:
    """Central configuration for the documented MPPT experiment.

    These values define the sparse scan, neural model, training loop, physics loss,
    local P&O refinement, and plotting limits. They are intentionally preserved so
    the documented notebook follows the same MPPT behavior as the original run.
    """
    seed: int = 7
    device: str = (
        "cuda"
        if torch.cuda.is_available()
        else ("mps" if torch.backends.mps.is_available() else "cpu")
    )

    # 12-point scan is intentionally preserved to emulate sparse MPPT measurement budget
    k_samples: int = 12
    sample_fracs_min: float = 0.05
    sample_fracs_max: float = 0.95

    # Sequence + TCNformer dimensions; these preserve the original hybrid model structure.
    n_zones: int = 4
    seq_len: int = 12
    seq_features: int = 4
    tcn_channels: int = 64
    tcn_layers: int = 3
    transformer_heads: int = 4
    transformer_dropout: float = 0.10
    transformer_ff_mult: int = 2
    latent_dim: int = 64
    dropout: float = 0.10

    # Training hyperparameters are kept unchanged for reproducible model behavior.
    batch_size: int = 128
    epochs: int = 1000
    lr: float = 1e-3
    weight_decay: float = 1e-2
    early_stop_patience: int = 20
    early_stop_min_delta: float = 1e-4
    label_smoothing: float = 0.03
    sim_holdout_fraction: float = 0.15

    # Loss weights balance supervised zone/offset/step learning with the physics term.
    w_data: float = 1.0
    w_physics: float = 0.25
    physics_weight_min_scale: float = 0.05
    physics_weight_max_scale: float = 5.0
    max_grad_norm: float = 1.0

    # Local P&O settings refine the neural seed without changing the global MPPT workflow.
    po_iterations: int = 14
    po_min_step_ratio: float = 0.003
    po_max_step_ratio: float = 0.050
    po_step_growth: float = 1.15
    po_step_shrink: float = 0.55
    po_stop_gain_ratio: float = 0.0005

    # evaluation / plots
    max_eval_curves: int = 2000
    n_viz: int = 8
    plot_peak_rel_thresh: float = 0.01
    plot_peak_min_gap: int = 20

    @property
    def sample_fracs(self) -> np.ndarray:
        return np.linspace(self.sample_fracs_min, self.sample_fracs_max, self.k_samples).astype(np.float32)

    @property
    def zone_edges(self) -> np.ndarray:
        return np.linspace(self.sample_fracs_min, self.sample_fracs_max, self.n_zones + 1).astype(np.float32)

cfg = Config()

# =========================================================


## Step 4: Reproducibility

This step sets Python, NumPy, and PyTorch random seeds and configures deterministic backend behavior where possible.

**Why this matters for partial-shading MPPT:** model training and validation splits involve randomness; fixed seeds help reviewers compare results across repeated notebook runs.

**What to check:** the selected compute device is printed, and repeated runs with the same environment should follow the same random path as before.


In [ ]:
# REPRODUCIBILITY
# =========================================================
def set_seed(seed: int) -> None:
    """Set Python, NumPy, and PyTorch seeds for repeatable training/evaluation.

    Reproducibility is important here because reviewer comparisons should reflect
    the same train/validation split, initialization path, and stochastic batches.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

set_seed(cfg.seed)
print("Device:", cfg.device)

# =========================================================


## Step 5: Dataset Upload and File Validation

This step preserves the original dataset path behavior: if `DATASET_PATH` is not set, the notebook attempts the Colab upload flow, then verifies that the selected file exists.

**Why this matters for partial-shading MPPT:** the MPPT workflow depends on a valid PV curve dataset before any cleaning, training, or evaluation can occur.

**What to check:** the printed dataset path should point to the intended `.mat` or `.npz` file.


In [ ]:
# COLAB FILE UPLOAD
# =========================================================
# If no path is provided, keep the original Colab upload workflow for the dataset.
if DATASET_PATH is None:
    try:
        from google.colab import files  # type: ignore
        uploaded = files.upload()
        if len(uploaded) == 0:
            raise RuntimeError("No file uploaded.")
        DATASET_PATH = next(iter(uploaded.keys()))
    except Exception as e:
        raise RuntimeError(
            "Set DATASET_PATH manually or run this in Colab and upload your .npz/.mat dataset."
        ) from e

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

print("Dataset path:", DATASET_PATH)

# =========================================================


## Step 6: Dataset Loading

This step defines helpers for traversing nested dataset containers, reading `.npz`/`.mat` files, inspecting MATLAB keys, and separating simulated, shaded experimental, and normal experimental curve groups.

**Why this matters for partial-shading MPPT:** MATLAB exports often store PV curves in nested object arrays or key-specific groups. Robust extraction is needed before the same MPPT algorithm can be applied to each curve.

**What to check:** when a `.mat` file is loaded, inspect the printed key summary and confirm that simulated and experimental curve counts are reasonable.


In [ ]:
# DATA LOADING
# =========================================================
def iter_curve_items(curves: Any):
    """Yield curve-like items from nested containers without assuming one storage schema.

    Parameters
    ----------
    curves : Any
        Container returned by dataset loaders (lists, tuples, ndarrays, MATLAB object arrays).

    Returns
    -------
    item : Any
        Individual candidate curve objects to be parsed by ``extract_vi``.

    Notes
    -----
    MATLAB exports often wrap curve arrays inside object matrices; this iterator normalizes
    traversal so downstream extraction is format-agnostic.
    """
    if isinstance(curves, list):
        for item in curves:
            yield item
        return
    if isinstance(curves, tuple):
        for item in curves:
            yield item
        return

    arr = np.asarray(curves, dtype=object)
    if arr.ndim == 0:
        yield arr.item()
        return

    if arr.ndim >= 3 and (arr.shape[-2] == 2 or arr.shape[-1] == 2):
        for k in range(arr.shape[0]):
            yield np.asarray(arr[k], dtype=float)
        return

    if arr.ndim == 2 and (arr.shape[0] == 2 or arr.shape[1] == 2):
        yield np.asarray(arr, dtype=float)
        return

    for item in arr.ravel():
        yield item


def _maybe_struct_get(obj: Any, names: Sequence[str]) -> Any:
    """Retrieve the first available field name from dicts or MATLAB-struct-like objects."""
    for name in names:
        if isinstance(obj, dict) and name in obj:
            return obj[name]
        if hasattr(obj, name):
            return getattr(obj, name)
    return None


def extract_vi(curve: Any) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    """Extract voltage and current/power vectors from heterogeneous curve encodings.

    Parameters
    ----------
    curve : Any
        Curve object that may be dict-like, MATLAB struct-like, numeric matrix, or nested object array.

    Returns
    -------
    v : Optional[np.ndarray]
        1-D voltage vector when extraction succeeds.
    y : Optional[np.ndarray]
        1-D second vector (current for I-V datasets, or power for direct P-V datasets).

    Notes
    -----
    This function intentionally accepts many storage layouts (2xN, Nx2, nested object wrappers)
    because raw .mat exports are inconsistent across acquisition pipelines.
    """
    if curve is None:
        return None, None

    if isinstance(curve, dict):
        v = _maybe_struct_get(curve, ["v", "V", "voltage", "x"])
        i = _maybe_struct_get(curve, ["i", "I", "current", "y"])
        if v is not None and i is not None:
            return np.asarray(v, dtype=float).ravel(), np.asarray(i, dtype=float).ravel()

    arr = np.asarray(curve, dtype=object)
    if arr.ndim == 0:
        try:
            return extract_vi(arr.item())
        except Exception:
            return None, None

    # numeric matrix path
    try:
        c = np.asarray(curve)
        if c.ndim == 2:
            if c.shape[0] == 2:
                return np.asarray(c[0], dtype=float).ravel(), np.asarray(c[1], dtype=float).ravel()
            if c.shape[1] >= 2:
                return np.asarray(c[:, 0], dtype=float).ravel(), np.asarray(c[:, 1], dtype=float).ravel()
    except Exception:
        pass

    # nested object wrapper path
    flat = arr.reshape(-1)
    for item in flat:
        try:
            v, i = extract_vi(item)
            if v is not None and i is not None:
                return v, i
        except Exception:
            continue
    return None, None


def _load_npz(path: str) -> Dict[str, Any]:
    """Load a NumPy archive while preserving the original curve-group keys."""
    """Load an NPZ dataset into a plain key/value dictionary."""
    d = np.load(path, allow_pickle=True)
    return {k: d[k] for k in d.files}


def _load_mat(path: str) -> Dict[str, Any]:
    """Load a MATLAB file and expose its keys for the configured PV curve groups."""
    """Load a MATLAB dataset and drop private ``__`` keys."""
    if loadmat is None:
        raise RuntimeError("scipy is required to read .mat files")
    m = loadmat(path)
    return {k: v for k, v in m.items() if not k.startswith("__")}


def _combine_curve_groups(raw: Dict[str, Any], keys: Sequence[str]) -> List[Any]:
    """Concatenate curves from selected MATLAB keys while tolerating missing groups."""
    rows: List[Any] = []
    for k in keys:
        if k not in raw:
            continue
        rows.extend(list(iter_curve_items(raw[k])))
    return rows


def load_dataset(path: str):
    """Load simulated/shaded/normal curve groups from .npz or .mat sources."""
    # NPZ files are expected to carry already-separated simulated/experimental groups.
    if path.lower().endswith(".npz"):
        raw = _load_npz(path)
        sim_curves = list(iter_curve_items(raw["sim_curves"])) if "sim_curves" in raw else []
        exp_sh_curves = list(iter_curve_items(raw["exp_sh_curves"])) if "exp_sh_curves" in raw else []
        exp_ok_curves = list(iter_curve_items(raw["exp_ok_curves"])) if "exp_ok_curves" in raw else []
        if len(exp_sh_curves) == 0 and "exp_curves" in raw:
            exp_sh_curves = list(iter_curve_items(raw["exp_curves"]))
        return sim_curves, exp_sh_curves, exp_ok_curves
    # MAT files are inspected because MATLAB exports can hide PV curves under several keys.
    if path.lower().endswith(".mat"):
        raw = _load_mat(path)
        print("\nMAT key inspection:")
        for k, v in raw.items():
            try:
                a = np.asarray(v, dtype=object)
                print(f"  - key={k}, type={type(v).__name__}, shape={a.shape}, dtype={a.dtype}")
            except Exception:
                print(f"  - key={k}, type={type(v).__name__}")
        sim_curves = _combine_curve_groups(raw, MAT_SIM_KEYS)
        exp_sh_curves = _combine_curve_groups(raw, MAT_EXP_SH_KEYS)
        exp_ok_curves = _combine_curve_groups(raw, MAT_EXP_OK_KEYS)
        print("\nSelected dataset sources:")
        print("  - selected simulated source keys:", MAT_SIM_KEYS)
        print("  - selected experimental shaded source keys:", MAT_EXP_SH_KEYS)
        print("  - selected experimental ok source keys:", MAT_EXP_OK_KEYS)
        print("  - number of simulated curves:", len(sim_curves))
        print("  - number of shaded experimental curves:", len(exp_sh_curves))
        print("  - number of ok experimental curves:", len(exp_ok_curves))
        return sim_curves, exp_sh_curves, exp_ok_curves
    raise ValueError("Dataset must be .npz or .mat")

# =========================================================


## Step 6A: Standardize Uploaded Curve Columns

`PV_Data_Reduced.mat` is handled with one dataset-level column format instead of per-curve guessing. For this dataset, `NEW_DATASET_FORMAT = "vp"`, so every curve is interpreted consistently as `[V, P]` before being converted to pseudo `[V, I]`.

The original MPPT pipeline expects curves in `[V, I]` format and later computes power as `P = V × I`. For direct P-V data, this step constructs a pseudo-current using `I = P / V`, allowing the downstream pipeline to keep computing `P = V × I` without changing the model, training process, evaluation logic, P&O refinement, plotting functions, or MPPT algorithm.

Using one explicit format for the full dataset avoids jagged/sawtooth P-V plots caused by noisy curve-by-curve automatic column detection and reduces output to a compact standardization summary.


In [ ]:
from typing import Any, List, Optional, Tuple
import numpy as np


def extract_curve_matrix(curve: Any) -> Optional[np.ndarray]:
    """
    Convert a raw curve into a clean 2-D numeric array.

    Supports:
    - N x 2
    - N x 3
    - 2 x N
    - 3 x N

    Returns an array shaped as N x C.
    """

    try:
        arr = np.asarray(curve, dtype=np.float64)
    except Exception:
        return None

    if arr.ndim != 2:
        return None

    # Convert C x N into N x C.
    if arr.shape[0] in (2, 3) and arr.shape[1] > arr.shape[0]:
        arr = arr.T

    if arr.shape[1] not in (2, 3):
        return None

    finite_rows = np.all(np.isfinite(arr), axis=1)
    arr = arr[finite_rows]

    if arr.shape[0] < 3:
        return None

    return arr


def convert_vp_to_vi(v: np.ndarray, p: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Convert direct P-V data into pseudo I-V data.

    This prevents the wrong operation:

        wrong_P = V * P

    After conversion, the existing pipeline can safely compute:

        P = V * I_pseudo
    """

    v = np.asarray(v, dtype=np.float64).ravel()
    p = np.asarray(p, dtype=np.float64).ravel()

    finite = np.isfinite(v) & np.isfinite(p)
    v = v[finite]
    p = p[finite]

    keep = v >= 0.0
    v = v[keep]
    p = p[keep]

    if len(v) < 3:
        return np.array([]), np.array([])

    idx = np.argsort(v)
    v = v[idx]
    p = p[idx]

    p = np.clip(p, 0.0, None)

    # Merge duplicate voltages using max power.
    unique_v = []
    unique_p = []

    start = 0
    for k in range(1, len(v)):
        if abs(v[k] - v[start]) > 1e-9:
            vg = v[start:k]
            pg = p[start:k]
            best = int(np.argmax(pg))
            unique_v.append(float(vg[best]))
            unique_p.append(float(pg[best]))
            start = k

    vg = v[start:]
    pg = p[start:]
    best = int(np.argmax(pg))
    unique_v.append(float(vg[best]))
    unique_p.append(float(pg[best]))

    v = np.asarray(unique_v, dtype=np.float64)
    p = np.asarray(unique_p, dtype=np.float64)

    if len(v) < 3:
        return np.array([]), np.array([])

    # Force P(0)=0.
    if v[0] > 1e-9:
        v = np.concatenate([[0.0], v])
        p = np.concatenate([[0.0], p])
    else:
        v[0] = 0.0
        p[0] = 0.0

    # Force P(Voc)=0. Use a slightly larger voltage if needed,
    # not the same voltage, to avoid duplicate endpoint problems.
    if p[-1] > 1e-6:
        positive_dv = np.diff(v)
        positive_dv = positive_dv[positive_dv > 1e-9]

        if len(positive_dv) > 0:
            dv = float(np.median(positive_dv))
        else:
            dv = max(float(v[-1]) * 1e-3, 1e-6)

        v = np.concatenate([v, [float(v[-1] + dv)]])
        p = np.concatenate([p, [0.0]])
    else:
        p[-1] = 0.0

    # Build pseudo-current.
    i = np.zeros_like(p)
    nonzero = v > 1e-9
    i[nonzero] = p[nonzero] / v[nonzero]

    # Estimate pseudo Isc at V=0.
    valid_i = i[np.isfinite(i) & (i > 0.0)]
    i[0] = float(np.percentile(valid_i, 85)) if len(valid_i) > 0 else 0.0

    i = np.clip(i, 0.0, None)

    return v.astype(np.float64), i.astype(np.float64)


def standardize_curve_by_format(
    curve: Any,
    curve_format: str,
) -> Optional[np.ndarray]:
    """
    Standardize one uploaded curve into [V, I] format.

    Supported curve_format values:
    - "vi"  = [V, I]
    - "vp"  = [V, P]
    - "vip" = [V, I, P]
    - "vpi" = [V, P, I]
    - "ivp" = [I, V, P]
    - "ipv" = [I, P, V]
    - "pvi" = [P, V, I]
    - "piv" = [P, I, V]

    The output is always 2 x N:
        row 0 = V
        row 1 = I_for_pipeline
    """

    arr = extract_curve_matrix(curve)
    if arr is None:
        return None

    fmt = curve_format.lower().strip()

    if arr.shape[1] != len(fmt):
        raise ValueError(
            f"Format '{curve_format}' has {len(fmt)} columns, "
            f"but curve has {arr.shape[1]} columns."
        )

    col_map = {name: idx for idx, name in enumerate(fmt)}

    if "v" not in col_map:
        raise ValueError(f"curve_format must include voltage column 'v'. Got: {curve_format}")

    v = arr[:, col_map["v"]]

    if "i" in col_map:
        i = arr[:, col_map["i"]]
        idx = np.argsort(v)
        return np.vstack([v[idx], i[idx]])

    if "p" in col_map:
        p = arr[:, col_map["p"]]
        v_out, i_out = convert_vp_to_vi(v, p)

        if len(v_out) < 3:
            return None

        return np.vstack([v_out, i_out])

    raise ValueError(f"curve_format must include either 'i' or 'p'. Got: {curve_format}")


def standardize_curves_by_format(
    curves: List[Any],
    curve_format: str,
    max_warnings: int = 5,
) -> List[np.ndarray]:
    """
    Standardize all curves using one dataset-level format.

    This avoids noisy and inconsistent per-curve guessing.
    """

    standardized = []
    skipped = 0

    for curve in curves:
        try:
            out = standardize_curve_by_format(curve, curve_format)
            if out is None:
                skipped += 1
                continue
            standardized.append(out)
        except Exception:
            skipped += 1
            if skipped <= max_warnings:
                print(f"Warning: skipped one curve during format standardization.")

    print(
        f"Curve standardization summary: "
        f"format='{curve_format}', kept={len(standardized)}, skipped={skipped}"
    )

    return standardized


## Step 7: PV Curve Extraction, Cleaning, and Endpoint Repair

This step defines the curve-cleaning helpers: duplicate-voltage handling, short-circuit-current estimation, open-circuit endpoint insertion, dense resampling, P=V×I conversion, validation, and peak diagnostics.

**Why this matters for partial-shading MPPT:** noisy or physically inconsistent I-V curves can create false peaks or interpolation artifacts. Enforcing endpoints and cleaning low-voltage artifacts helps the sparse scan and dense true-MPP label refer to a physically meaningful PV curve.

**What to check:** later preprocessing summaries should show how many curves were accepted/skipped and whether endpoint/duplicate repairs occurred.


In [ ]:
# CLEANING / VALIDATION (STRICT + PHYSICALLY CONSISTENT)
# =========================================================
# Dense sweeps may contain almost identical voltages; merge them so downstream checks
# enforce strictly increasing voltage without rejecting valid dense acquisitions.
def merge_near_duplicate_voltage(v: np.ndarray, i: np.ndarray, rel_tol: float = 1e-7, abs_tol: float = 1e-6) -> Tuple[np.ndarray, np.ndarray]:
    """Merge repeated or nearly repeated voltage samples by averaging current values.

    Duplicate voltage entries can destabilize interpolation, so this helper keeps a
    single representative current at each voltage without changing the intended PV curve.
    """
    """Merge near-duplicate voltage samples instead of rejecting dense clean curves.

    Reason: experimental dense PV sweeps can have dV around 1e-8, which is valid and
    should be merged for robust downstream validation.
    """
    v = np.asarray(v, dtype=np.float64).ravel()
    i = np.asarray(i, dtype=np.float64).ravel()
    if len(v) != len(i) or len(v) == 0:
        return np.array([], dtype=np.float64), np.array([], dtype=np.float64)

    idx = np.argsort(v)
    v = v[idx]
    i = i[idx]
    voc = float(np.max(v)) if len(v) else 0.0
    group_tol = max(float(abs_tol), float(rel_tol) * max(voc, 0.0))

    out_v, out_i = [], []
    g_v, g_i = [float(v[0])], [float(i[0])]
    for k in range(1, len(v)):
        if float(v[k] - g_v[-1]) <= group_tol:
            g_v.append(float(v[k]))
            g_i.append(float(i[k]))
        else:
            out_v.append(float(np.mean(g_v)))
            out_i.append(float(np.median(g_i)))  # median current is robust to local outliers within repeated-voltage groups
            g_v, g_i = [float(v[k])], [float(i[k])]
    out_v.append(float(np.mean(g_v)))
    out_i.append(float(np.median(g_i)))  # median current is robust to local outliers within repeated-voltage groups
    return np.asarray(out_v, dtype=np.float64), np.asarray(out_i, dtype=np.float64)


def estimate_isc_from_low_voltage_window(v: np.ndarray, i: np.ndarray, low_frac: float = 0.08) -> float:
    """Estimate short-circuit current from the low-voltage portion of an I-V curve.

    The estimate supports endpoint repair so the curve begins near the physical
    I(0)=Isc condition used by the MPPT feature builder.
    """
    """Estimate Isc from low-voltage window, not only first few samples.

    Reason: first points can be noisy/collapsed and can cause inward low-voltage P-V truncation.
    """
    v = np.asarray(v, dtype=np.float64).ravel()
    i = np.asarray(i, dtype=np.float64).ravel()
    if len(v) != len(i) or len(v) == 0:
        return 0.0
    finite_pos = np.isfinite(i) & (i > 0.0)
    if not np.any(finite_pos):
        return 0.0
    voc = float(np.max(v)) if len(v) else 0.0
    low_mask = finite_pos & np.isfinite(v) & (v <= low_frac * max(voc, 0.0))
    if int(np.sum(low_mask)) >= 5:
        return float(np.percentile(i[low_mask], 85))
    return float(np.percentile(i[finite_pos], 85))


# Repair endpoints to satisfy PV physics before feature extraction/training:
# I(0)≈Isc and I(Voc)=0, which implies P(0)=P(Voc)=0 after P=V*I conversion.
def enforce_physical_iv_endpoints(v: np.ndarray, i: np.ndarray, voc_est: Optional[float] = None, diag: Optional[Dict[str, int]] = None, tol: float = 1e-9) -> Tuple[np.ndarray, np.ndarray]:
    v = np.asarray(v, dtype=np.float64).ravel()
    i = np.asarray(i, dtype=np.float64).ravel()
    if len(v) != len(i) or len(v) < 2:
        return np.array([]), np.array([])

    good = np.isfinite(v) & np.isfinite(i)
    v, i = v[good], i[good]
    if len(v) < 2:
        return np.array([]), np.array([])

    idx = np.argsort(v)
    v, i = v[idx], i[idx]
    v, i = merge_near_duplicate_voltage(v, i)
    if len(v) < 2:
        return np.array([]), np.array([])

    voc_ref = float(max(np.max(v), float(voc_est) if voc_est is not None else np.max(v)))
    isc_est = max(estimate_isc_from_low_voltage_window(v, i), 0.0)

    low_region = v <= (0.15 * max(voc_ref, 0.0))
    bad_low_current = low_region & (i < 0.35 * isc_est)
    removed = int(np.sum(bad_low_current))
    if removed > 0:
        v, i = v[~bad_low_current], i[~bad_low_current]
    if diag is not None:
        diag['low_voltage_artifact_removed'] = int(diag.get('low_voltage_artifact_removed', 0)) + removed
    if len(v) < 2:
        return np.array([]), np.array([])

    if abs(float(v[0])) > tol:
        v = np.concatenate(([0.0], v))
        i = np.concatenate(([isc_est], i))
        if diag is not None:
            diag['left_endpoint_inserted'] = int(diag.get('left_endpoint_inserted', 0)) + 1
    else:
        v[0] = 0.0
        if float(i[0]) < isc_est:
            i[0] = isc_est
            if diag is not None:
                diag['left_endpoint_repaired'] = int(diag.get('left_endpoint_repaired', 0)) + 1

    voc_target = max(voc_ref, float(v[-1]))
    if not (abs(float(v[-1]) - voc_target) <= tol and abs(float(i[-1])) <= tol):
        if diag is not None:
            diag['voc_endpoint_inserted'] = int(diag.get('voc_endpoint_inserted', 0)) + 1
    pre = v < (voc_target - tol)
    v = np.concatenate([v[pre], [voc_target]])
    i = np.concatenate([np.clip(i[pre], 0.0, None), [0.0]])

    v, i = merge_near_duplicate_voltage(v, i)
    if len(v) < 8:
        return np.array([]), np.array([])
    v[0] = 0.0
    i[0] = max(float(i[0]), isc_est)
    i[-1] = 0.0
    if np.any(np.diff(v) <= 0.0):
        return np.array([]), np.array([])
    return v.astype(np.float64), np.clip(i, 0.0, None).astype(np.float64)


# Interpolate cleaned curves onto a uniform voltage grid so MPP/peaks/plots are
# comparable and not biased by irregular experimental sample spacing.
def resample_iv_curve(v: np.ndarray, i: np.ndarray, n_points: int = 2000) -> Tuple[np.ndarray, np.ndarray]:
    """Interpolate a cleaned I-V curve onto a dense voltage grid for MPP search."""
    v = np.asarray(v, dtype=np.float64).ravel()
    i = np.asarray(i, dtype=np.float64).ravel()
    if len(v) != len(i) or len(v) < 2 or int(n_points) < 2:
        return np.array([]), np.array([])
    voc = float(np.max(v))
    if voc <= 0.0:
        return np.array([]), np.array([])
    v_grid = np.linspace(0.0, voc, int(n_points), dtype=np.float64)
    i_grid = np.interp(v_grid, v, i)
    i_grid = np.clip(i_grid, 0.0, None)
    i_grid[0] = max(float(i_grid[0]), estimate_isc_from_low_voltage_window(v, i))
    i_grid[-1] = 0.0
    return v_grid.astype(np.float64), i_grid.astype(np.float64)


def build_pv_curve_from_iv(v: np.ndarray, i: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Convert current to power with P=V*I after endpoint-aware I-V cleaning."""
    v_grid, i_grid = resample_iv_curve(v, i, n_points=2000)
    if len(v_grid) < 2:
        return np.array([]), np.array([])
    p_grid = np.clip(v_grid * i_grid, 0.0, None)
    p_grid[0] = 0.0
    p_grid[-1] = 0.0
    return v_grid.astype(np.float64), p_grid.astype(np.float64)


# Strict cleaner for true I-V data: sanitize, sign-correct, keep physical quadrant,
# merge near-duplicates, estimate Voc region, and enforce physically consistent endpoints.
def clean_iv_curve_strict(v: np.ndarray, i: np.ndarray, diag: Optional[Dict[str, int]] = None) -> Tuple[np.ndarray, np.ndarray]:
    """Clean raw I-V vectors while preserving the original PV-physics repair rules.

    The routine removes non-finite samples, handles sign/orientation issues, suppresses
    low-voltage artifacts, merges duplicate voltages, and calls endpoint repair before
    dense MPP computation.
    """
    v = np.asarray(v, dtype=np.float64).ravel()
    i = np.asarray(i, dtype=np.float64).ravel()
    if len(v) != len(i) or len(v) < 3:
        return np.array([]), np.array([])

    good = np.isfinite(v) & np.isfinite(i)
    v, i = v[good], i[good]
    if len(v) < 3:
        return np.array([]), np.array([])

    if float(np.nanmedian(i)) < 0.0:
        i = -i
        if diag is not None:
            diag['sign_flip'] = int(diag.get('sign_flip', 0)) + 1

    quad = (v >= 0.0) & (i >= 0.0)
    v, i = v[quad], i[quad]
    if len(v) < 3:
        return np.array([]), np.array([])

    idx = np.argsort(v)
    v, i = v[idx], i[idx]
    v, i = merge_near_duplicate_voltage(v, i)
    if len(v) < 3:
        return np.array([]), np.array([])

    p = v * i
    valid = (i > 1e-9) & (p > 1e-12)
    if not np.any(valid):
        return np.array([]), np.array([])
    last_pos_idx = int(np.where(valid)[0][-1])

    if last_pos_idx + 1 < len(v):
        v1, i1 = float(v[last_pos_idx]), float(i[last_pos_idx])
        v2, i2 = float(v[last_pos_idx + 1]), float(i[last_pos_idx + 1])
    elif len(v) >= 2:
        v1, i1 = float(v[-2]), float(v[-2]*0 + i[-2])
        v2, i2 = float(v[-1]), float(i[-1])
    else:
        return np.array([]), np.array([])
    denom = (v2 - v1)
    voc_est = float(v[last_pos_idx])
    if abs(denom) > 1e-12:
        slope = (i2 - i1) / denom
        if abs(slope) > 1e-12:
            voc_est = float(v1 - i1 / slope)
    voc_est = max(voc_est, float(v[last_pos_idx]))

    v_clean, i_clean = enforce_physical_iv_endpoints(v[:last_pos_idx+1], i[:last_pos_idx+1], voc_est=voc_est, diag=diag)
    if len(v_clean) < 8:
        return np.array([]), np.array([])
    return v_clean.astype(np.float64), i_clean.astype(np.float64)


# Validation rejects only non-positive voltage steps (not tiny positive spacing) and
# enforces finite/nonnegative/positive-power physical plausibility checks.
def validate_cleaned_curve(v: np.ndarray, i: np.ndarray, tol: float = 1e-6) -> bool:
    """Check that cleaned I-V samples are usable for interpolation and MPPT labeling."""
    v = np.asarray(v, dtype=np.float64).ravel()
    i = np.asarray(i, dtype=np.float64).ravel()
    if len(v) != len(i) or len(v) < 8:
        return False
    if not np.all(np.isfinite(v)) or not np.all(np.isfinite(i)):
        return False
    if np.any(np.diff(v) <= 0.0):
        return False
    if not np.isclose(float(v[0]), 0.0, atol=tol):
        return False
    if not np.isclose(float(i[-1]), 0.0, atol=tol):
        return False
    if float(i[0]) <= tol:
        return False
    if np.any(i < -tol):
        return False
    if float(np.max(v)) <= tol:
        return False
    if float(np.max(v * i)) <= tol:
        return False
    return True


def compute_mpp_dense(v: np.ndarray, i: np.ndarray, n: int = 2000) -> Tuple[float, float, np.ndarray, np.ndarray]:
    """Find the dense-grid true MPP used as the supervised target/reference."""
    v_res, p_res = build_pv_curve_from_iv(v, i)
    if len(v_res) == 0:
        return 0.0, 0.0, np.array([]), np.array([])
    v_dense, p_dense = v_res, p_res
    if int(n) > 1 and int(n) != len(v_res):
        v_dense = np.linspace(float(v_res[0]), float(v_res[-1]), int(n), dtype=np.float64)
        p_dense = np.interp(v_dense, v_res, p_res)
    k = int(np.argmax(p_dense))
    return float(v_dense[k]), float(p_dense[k]), v_dense.astype(np.float64), p_dense.astype(np.float64)


def prepare_pv_plot_curve(v: np.ndarray, i: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Prepare visualization P-V only from cleaned, uniformly resampled I-V."""
    pv_v, pv_p = build_pv_curve_from_iv(v, i)
    if len(pv_v) < 2:
        return np.array([]), np.array([])
    assert np.isclose(float(pv_p[0]), 0.0, atol=1e-9), 'derived P-V must start at 0 W'
    assert np.isclose(float(pv_p[-1]), 0.0, atol=1e-9), 'derived P-V must end at 0 W'
    return pv_v.astype(np.float64), pv_p.astype(np.float64)


def diagnose_cleaning_one_curve(curve):
    v_raw, i_raw = extract_vi(curve)
    print(f'raw shape: v={np.shape(v_raw)}, i={np.shape(i_raw)}')
    v_clean, i_clean = clean_iv_curve_strict(v_raw, i_raw)
    print(f'cleaned shape: v={v_clean.shape}, i={i_clean.shape}')
    print(f'len match: {len(v_clean) == len(i_clean)}')
    print(f'all finite: {np.all(np.isfinite(v_clean)) and np.all(np.isfinite(i_clean))}')
    d = np.diff(v_clean) if len(v_clean) > 1 else np.array([])
    print(f'min diff(v_clean): {float(np.min(d)) if len(d) else np.nan}')
    print(f'strictly increasing v (diff > 0.0): {bool(np.all(d > 0.0)) if len(d) else False}')
    print(f'v[0]: {float(v_clean[0]) if len(v_clean) else np.nan}')
    print(f'i[-1]: {float(i_clean[-1]) if len(i_clean) else np.nan}')
    print(f'max power: {float(np.max(v_clean * i_clean)) if len(v_clean) else 0.0}')
    print(f'validate_cleaned_curve: {validate_cleaned_curve(v_clean, i_clean)}')


def dense_peak_profile_for_plot(oracle, n_dense: int = 2000, rel_thresh: float = 0.01, min_index_gap: int = 20) -> Dict[str, Any]:
    v, i = oracle.curve_for_plot()
    pv_v, pv_p = prepare_pv_plot_curve(v, i)
    if len(pv_v) < 3:
        return {
            "n_peaks": 0,
            "peak_idxs": [],
            "peak_prominence_score": 0.0,
            "v_dense": np.array([]),
            "p_dense": np.array([]),
        }
    v_dense = np.linspace(float(np.min(pv_v)), float(np.max(pv_v)), n_dense)
    p_dense = np.interp(v_dense, pv_v, pv_p)
    n_peaks, peak_idxs = count_local_maxima_dense(p_dense, rel_thresh=rel_thresh, min_index_gap=min_index_gap)
    if n_peaks < 2:
        prom_score = 0.0
    else:
        peak_vals = sorted([float(p_dense[k]) for k in peak_idxs], reverse=True)
        prom_score = float(peak_vals[1] / max(peak_vals[0], 1e-12))
    return {
        "n_peaks": int(n_peaks),
        "peak_idxs": list(peak_idxs),
        "peak_prominence_score": float(prom_score),
        "v_dense": v_dense.astype(np.float32),
        "p_dense": p_dense.astype(np.float32),
    }


def count_local_maxima_dense(p_arr: np.ndarray, rel_thresh: float = 0.01, min_index_gap: int = 20) -> Tuple[int, List[int]]:
    p = np.asarray(p_arr, dtype=float).ravel()
    if len(p) < 5:
        return 0, []
    vmax = max(float(np.max(p)), 1e-12)
    thresh = rel_thresh * vmax
    peaks: List[int] = []
    for k in range(1, len(p) - 1):
        if p[k] >= p[k - 1] and p[k] >= p[k + 1] and p[k] >= thresh:
            peaks.append(k)
    if len(peaks) <= 1:
        return len(peaks), peaks
    merged = [peaks[0]]
    for idx in peaks[1:]:
        if idx - merged[-1] < min_index_gap:
            if p[idx] > p[merged[-1]]:
                merged[-1] = idx
        else:
            merged.append(idx)
    return len(merged), merged


def dense_peak_profile(v: np.ndarray, i: np.ndarray, cfg: Config) -> Dict[str, Any]:
    vmpp, pmpp, v_dense, p_dense = compute_mpp_dense(v, i)
    if len(v_dense) == 0:
        return {"vmpp": 0.0, "pmpp": 0.0, "n_peaks": 0, "peak_idxs": [], "peak_prominence_score": 0.0,
                "v_dense": np.array([]), "p_dense": np.array([])}
    n_peaks, peak_idxs = count_local_maxima_dense(
        p_dense, rel_thresh=cfg.plot_peak_rel_thresh, min_index_gap=cfg.plot_peak_min_gap
    )
    if n_peaks < 2:
        prom_score = 0.0
    else:
        peak_vals = sorted([float(p_dense[k]) for k in peak_idxs], reverse=True)
        prom_score = float(peak_vals[1] / max(peak_vals[0], 1e-12))
    return {
        "vmpp": vmpp,
        "pmpp": pmpp,
        "n_peaks": int(n_peaks),
        "peak_idxs": list(peak_idxs),
        "peak_prominence_score": prom_score,
        "v_dense": v_dense,
        "p_dense": p_dense,
    }

# =========================================================
# CURVE ORACLE + FEATURES
# =========================================================


## Step 8: Sparse 12-Point Feature Sampling and Dataset Rows

This step builds the `CurveOracle`, MPP zone/offset labels, sparse voltage/current/power features, dense peak profiles, and row dictionaries used for model training and evaluation.

**Why this matters for partial-shading MPPT:** measuring a full dense curve is expensive. The 12-point sparse scan emulates a limited practical measurement budget while still giving the model information about multi-peak P-V shape.

**What to check:** each accepted row should contain sparse sequence features, normalized global features, a true MPP voltage ratio, zone label, offset label, and P&O step target.


In [ ]:
class CurveOracle:
    """Cleaned PV curve wrapper used for interpolation, labels, and local MPPT checks.

    The oracle centralizes Voc, true MPP, dense samples, and power-at-voltage queries so
    feature creation, evaluation, plotting, and P&O refinement all use the same curve.
    """
    """Measurement interface over one cleaned I-V curve for controller-style voltage queries.

    Notes
    -----
    The oracle stores cleaned data, computes Voc/Isc/true MPP, and exposes interpolation methods
    that emulate how an MPPT controller samples the plant at chosen voltages.
    """
    def __init__(self, curve: Any):
        v_raw, i_raw = extract_vi(curve)
        self._v, self._i = clean_iv_curve_strict(v_raw, i_raw)
        if not validate_cleaned_curve(self._v, self._i):
            self._v = np.array([], dtype=np.float32)
            self._i = np.array([], dtype=np.float32)
            self.voc = 0.0
            self.isc = 0.0
            self.vmpp_true = 0.0
            self.pmpp_true = 0.0
            self.v_dense = np.array([])
            self.p_dense = np.array([])
            self.dense_peak_count = 0
            self.peak_prominence_score = 0.0
        else:
            self.voc = float(np.max(self._v))
            self.isc = float(np.interp(float(np.min(self._v)), self._v, self._i)) if len(self._v) else 0.0
            prof = dense_peak_profile(self._v, self._i, cfg)
            self.vmpp_true = float(prof["vmpp"])
            self.pmpp_true = float(prof["pmpp"])
            self.v_dense = prof["v_dense"]
            self.p_dense = prof["p_dense"]
            self.dense_peak_count = int(prof["n_peaks"])
            self.peak_prominence_score = float(prof["peak_prominence_score"])

    def measure(self, vq: float) -> float:
        if self.voc <= 0 or len(self._v) < 2:
            return 0.0
        vq = float(np.clip(vq, float(np.min(self._v)), self.voc))
        return float(np.interp(vq, self._v, self._i))

    def power_at(self, vq: float) -> float:
        return float(vq * self.measure(vq))

    def curve_for_plot(self) -> Tuple[np.ndarray, np.ndarray]:
        return self._v.copy(), self._i.copy()


def zone_index_and_offset(vhat_true: float, zone_edges: np.ndarray) -> Tuple[int, float]:
    z = int(np.searchsorted(zone_edges[1:-1], vhat_true, side='right'))
    z = int(np.clip(z, 0, len(zone_edges) - 2))
    lo = float(zone_edges[z])
    hi = float(zone_edges[z + 1])
    center = 0.5 * (lo + hi)
    halfw = max(0.5 * (hi - lo), 1e-6)
    offset = float(np.clip((vhat_true - center) / halfw, -1.0, 1.0))
    return z, offset


# Build sparse-observation features: the model sees 12 query points, not full dense curves.
def build_feature_vector(v: np.ndarray, i: np.ndarray, sample_fracs: np.ndarray, cfg: Config) -> Dict[str, Any]:
    """Create the sparse 12-point scan features and MPP zone/offset labels.

    The network receives normalized voltage/current/power observations from the sparse
    scan plus compact global descriptors, matching a practical low-measurement MPPT pass.
    """
    voc = float(np.max(v))
    vmpp_true, pmpp_true, _, _ = compute_mpp_dense(v, i)
    true_vhat = float(np.clip(vmpp_true / max(voc, 1e-9), cfg.sample_fracs_min, cfg.sample_fracs_max))

    vq = sample_fracs * voc
    iq = np.interp(vq, v, i)
    p = v * i
    p_dense_max = max(float(np.max(p)), 1e-9)
    isc_ref = max(float(np.max(np.abs(iq))), 1e-9)

    v_norm = (vq / max(voc, 1e-9)).astype(np.float32)
    i_norm = (iq / isc_ref).astype(np.float32)
    p_norm = ((vq * iq) / p_dense_max).astype(np.float32)
    dpdv = np.diff(p_norm) / (np.diff(v_norm) + 1e-12)
    if len(dpdv) < len(v_norm):
        dpdv = np.pad(dpdv.astype(np.float32), (0, len(v_norm) - len(dpdv)), mode='edge')

    order_desc = np.argsort(p_norm)[::-1]
    best_idx = int(order_desc[0])
    second_idx = int(order_desc[1]) if len(order_desc) > 1 else best_idx
    top_gap_ratio = float((p_norm[best_idx] - p_norm[second_idx]) / max(abs(float(p_norm[best_idx])), 1e-9))

    dense_prof = dense_peak_profile(v, i, cfg)
    multi_peak_count = float(dense_prof["n_peaks"])
    prom_score = float(dense_prof["peak_prominence_score"])

    hist_extra = np.array([
        float(v_norm[best_idx]),
        float(best_idx / max(len(v_norm) - 1, 1)),
        top_gap_ratio,
        multi_peak_count,
    ], dtype=np.float32)
    env_feats = np.concatenate([
        hist_extra,
        np.array([
            np.log1p(voc),
            np.log1p(max(np.max(np.abs(i)), 1e-9)),
            np.log1p(max(pmpp_true, 1e-9)),
            prom_score,
        ], dtype=np.float32),
    ], axis=0).astype(np.float32)

    historical = np.stack([v_norm, i_norm, p_norm, dpdv.astype(np.float32)], axis=1).flatten().astype(np.float32)
    x = np.concatenate([historical, env_feats], axis=0).astype(np.float32)
    y_zone, y_offset = zone_index_and_offset(true_vhat, cfg.zone_edges)

    # heuristic target for the ANN step-size head: bigger if multi-peak / ambiguous
    y_step_ratio = float(np.clip(0.010 + 0.008 * min(multi_peak_count, 3.0) + 0.020 * max(0.08 - top_gap_ratio, 0.0),
                                 cfg.po_min_step_ratio, cfg.po_max_step_ratio))

    return {
        "x": x,
        "true_vhat": np.float32(true_vhat),
        "true_pmpp": np.float32(pmpp_true),
        "y_zone": np.int64(y_zone),
        "y_offset": np.float32(y_offset),
        "y_step_ratio": np.float32(y_step_ratio),
        "sparse_vhat": v_norm.astype(np.float32),
        "sparse_ihat": i_norm.astype(np.float32),
        "sparse_phat": p_norm.astype(np.float32),
        "dense_vhat": np.linspace(cfg.sample_fracs_min, cfg.sample_fracs_max, 256, dtype=np.float32),
        "dense_pratio": np.interp(
            np.linspace(cfg.sample_fracs_min, cfg.sample_fracs_max, 256, dtype=np.float32),
            dense_prof["v_dense"] / max(voc, 1e-9) if len(dense_prof["v_dense"]) else np.array([cfg.sample_fracs_min, cfg.sample_fracs_max], dtype=np.float32),
            dense_prof["p_dense"] / max(pmpp_true, 1e-9) if len(dense_prof["p_dense"]) else np.array([0.0, 0.0], dtype=np.float32),
            left=0.0,
            right=0.0,
        ).astype(np.float32),
        "coarse_best_vhat": np.float32(float(v_norm[best_idx])),
        "coarse_best_phat": np.float32(float(p_norm[best_idx]) * p_dense_max),
        "multi_peak_count": np.float32(multi_peak_count),
        "peak_prominence_score": np.float32(prom_score),
    }


def build_dataset_from_curves(curves, cfg: Config):
    """Convert a collection of raw curves into train/evaluation rows with diagnostics."""
    rows: List[Dict[str, Any]] = []
    stats = {
        "total": 0,
        "valid": 0,
        "skipped_extract": 0,
        "skipped_clean": 0,
        "skipped_validate": 0,
        "sign_flip": 0,
        "left_endpoint_inserted": 0,
        "left_endpoint_repaired": 0,
        "voc_endpoint_inserted": 0,
        "low_voltage_artifact_removed": 0,
        "rejected": 0,
    }
    for item in iter_curve_items(curves):
        stats["total"] += 1
        v_raw, i_raw = extract_vi(item)
        if v_raw is None or i_raw is None:
            stats["skipped_extract"] += 1
            stats["rejected"] += 1
            continue
        v, i = clean_iv_curve_strict(v_raw, i_raw, diag=stats)
        if len(v) < 8:
            stats["skipped_clean"] += 1
            stats["rejected"] += 1
            continue
        if not validate_cleaned_curve(v, i):
            stats["skipped_validate"] += 1
            stats["rejected"] += 1
            continue
        feat = build_feature_vector(v, i, cfg.sample_fracs, cfg)
        feat["curve"] = item
        rows.append(feat)
        stats["valid"] += 1
    print(
        "Preprocess summary:",
        {
            "sign_flip": stats["sign_flip"],
            "left_endpoint_inserted": stats["left_endpoint_inserted"],
            "left_endpoint_repaired": stats["left_endpoint_repaired"],
            "voc_endpoint_inserted": stats["voc_endpoint_inserted"],
            "low_voltage_artifact_removed": stats["low_voltage_artifact_removed"],
            "rejected": stats["rejected"],
        },
    )
    return rows, stats

# =========================================================
# MODEL
# =========================================================


## Step 9: Model Architecture and Physics-Informed Loss

This step defines the causal convolution block, Physics-Informed TCNformer, prediction decoding, interpolation utility, and combined zone/offset/step plus physics loss.

**Why this matters for partial-shading MPPT:** the model predicts a voltage zone and intra-zone offset so the controller starts near the likely global MPP instead of blindly following a local slope. The physics term keeps the prediction connected to the MPP condition on the sampled curve.

**What to check:** the model input dimension should match the sparse scan features plus global descriptors, and training logs should report both data and physics loss terms.


In [ ]:
class CausalConvBlock(nn.Module):
    """Causal/dilated residual block to extract local sequential patterns from sparse scans."""
    def __init__(self, channels: int, kernel_size: int, dilation: int, dropout: float):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(channels, channels, kernel_size=kernel_size, dilation=dilation)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = nn.functional.pad(x, (self.pad, 0))
        h = self.conv(h)
        h = self.act(h)
        h = self.drop(h)
        return x + h


class PhysicsInformedTCNformer(nn.Module):
    """Hybrid TCN+Transformer regressor/classifier for MPP seed prediction.

    Input = flattened (seq_len=12, seq_features=4) sparse sequence + 8 global features.
    TCN captures local voltage-axis structure, attention captures point-to-point dependencies,
    and fusion layers output: zone probabilities, intra-zone offset, and P&O step ratio.
    """
    def __init__(self, in_dim: int, n_zones: int, cfg: Config):
        super().__init__()
        self.cfg = cfg
        expected_in_dim = cfg.seq_len * cfg.seq_features + 8
        if in_dim != expected_in_dim:
            raise ValueError(f"Expected input dim={expected_in_dim}, got {in_dim}")
        self.seq_len = cfg.seq_len
        self.seq_features = cfg.seq_features

        self.seq_proj = nn.Conv1d(cfg.seq_features, cfg.tcn_channels, kernel_size=1)
        self.tcn = nn.ModuleList(
            [CausalConvBlock(cfg.tcn_channels, kernel_size=3, dilation=2 ** i, dropout=cfg.dropout) for i in range(cfg.tcn_layers)]
        )
        self.attn = nn.MultiheadAttention(
            embed_dim=cfg.tcn_channels,
            num_heads=cfg.transformer_heads,
            dropout=cfg.transformer_dropout,
            batch_first=True,
        )
        self.ffn = nn.Sequential(
            nn.Linear(cfg.tcn_channels, cfg.transformer_ff_mult * cfg.tcn_channels),
            nn.GELU(),
            nn.Dropout(cfg.transformer_dropout),
            nn.Linear(cfg.transformer_ff_mult * cfg.tcn_channels, cfg.tcn_channels),
        )
        self.norm1 = nn.LayerNorm(cfg.tcn_channels)
        self.norm2 = nn.LayerNorm(cfg.tcn_channels)
        self.temporal_ln = nn.LayerNorm(cfg.tcn_channels)
        self.fusion = nn.Sequential(
            nn.Linear(cfg.tcn_channels + 8, cfg.latent_dim),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.LayerNorm(cfg.latent_dim),
        )
        self.zone_head = nn.Linear(cfg.latent_dim, n_zones)
        self.offset_head = nn.Linear(cfg.latent_dim, 1)
        self.step_head = nn.Linear(cfg.latent_dim, 1)

    def forward(self, x: torch.Tensor):
        """Predict MPP zone probabilities, intra-zone offset, and P&O step ratio."""
        # Split flattened input into sequential scan features and global/environment features.
        hist = x[:, : self.seq_len * self.seq_features]
        env = x[:, self.seq_len * self.seq_features :]
        seq = hist.view(x.shape[0], self.seq_len, self.seq_features).transpose(1, 2)  # (B, F, L)
        h = self.seq_proj(seq)
        for block in self.tcn:
            h = block(h)
        h = h.transpose(1, 2)  # (B, L, C)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        h = self.norm1(h + attn_out)
        ffn_out = self.ffn(h)
        h = self.norm2(h + ffn_out)
        h = self.temporal_ln(h.mean(dim=1))
        fused = self.fusion(torch.cat([h, env], dim=1))

        zone_logits = self.zone_head(fused)
        offset_raw = self.offset_head(fused).squeeze(-1)
        step_raw = self.step_head(fused).squeeze(-1)
        zone_probs = nn.functional.softmax(zone_logits, dim=1)
        offset = torch.tanh(offset_raw)
        step_ratio_raw = torch.sigmoid(step_raw)
        step_ratio = self.cfg.po_min_step_ratio + (self.cfg.po_max_step_ratio - self.cfg.po_min_step_ratio) * step_ratio_raw
        return zone_probs, offset, step_ratio


def compute_pred_vhat(zone_probs: torch.Tensor, offset: torch.Tensor, zone_edges_t: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Map zone probabilities + intra-zone offset into normalized voltage prediction."""
    zone_centers = 0.5 * (zone_edges_t[:-1] + zone_edges_t[1:])
    zone_widths = zone_edges_t[1:] - zone_edges_t[:-1]
    pred_center = torch.sum(zone_probs * zone_centers.unsqueeze(0), dim=1)
    pred_width = torch.sum(zone_probs * zone_widths.unsqueeze(0), dim=1)
    pred_vhat = pred_center + 0.5 * pred_width * offset
    pred_vhat = torch.clamp(pred_vhat, float(zone_edges_t[0]), float(zone_edges_t[-1]))
    zone_conf = torch.max(zone_probs, dim=1).values
    return pred_vhat, zone_probs, pred_center, zone_conf


def interp_batch_1d(v_query: torch.Tensor, v_grid: torch.Tensor, y_grid: torch.Tensor) -> torch.Tensor:
    """Batch linear interpolation used by the physics term near predicted operating points."""
    idx_hi = torch.searchsorted(v_grid, v_query.unsqueeze(-1), right=False).squeeze(-1)
    idx_hi = torch.clamp(idx_hi, 1, v_grid.shape[1] - 1)
    idx_lo = idx_hi - 1
    b = torch.arange(v_grid.shape[0], device=v_grid.device)
    v0 = v_grid[b, idx_lo]
    v1 = v_grid[b, idx_hi]
    y0 = y_grid[b, idx_lo]
    y1 = y_grid[b, idx_hi]
    t = (v_query - v0) / torch.clamp(v1 - v0, min=1e-6)
    return y0 + t * (y1 - y0)


def pinn_zone_offset_loss(zone_probs, offset_pred, step_ratio_pred, batch, cfg: Config, zone_edges_t: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, float]]:
    """Compute the unchanged supervised + physics-informed training objective."""
    y_zone = batch["y_zone"]
    y_offset = batch["y_offset"]
    y_step_ratio = batch["y_step_ratio"]
    seq_vhat = batch["seq_vhat"]
    seq_ihat = batch["seq_ihat"]

    pred_vhat, zone_probs, _, _ = compute_pred_vhat(zone_probs, offset_pred, zone_edges_t)
    pred_vhat = pred_vhat.requires_grad_(True)
    zone_loss = nn.functional.nll_loss(torch.log(torch.clamp(zone_probs, min=1e-8)), y_zone)
    offset_loss = nn.functional.smooth_l1_loss(offset_pred, y_offset)
    step_loss = nn.functional.smooth_l1_loss(step_ratio_pred, y_step_ratio)

    i_pred = interp_batch_1d(pred_vhat, seq_vhat, seq_ihat)
    dI_dV = torch.autograd.grad(
        outputs=i_pred,
        inputs=pred_vhat,
        grad_outputs=torch.ones_like(i_pred),
        create_graph=True,
    )[0]
    physics_term = (dI_dV + (i_pred / torch.clamp(pred_vhat, min=1e-6))) ** 2
    physics_loss = torch.mean(physics_term)

    data_loss = 1.0 * zone_loss + 0.5 * offset_loss + 0.5 * step_loss
    relative_scale = (data_loss.detach() / torch.clamp(physics_loss.detach(), min=1e-8)).clamp(
        cfg.physics_weight_min_scale,
        cfg.physics_weight_max_scale,
    )
    w_physics_dynamic = cfg.w_physics * relative_scale
    total = cfg.w_data * data_loss + w_physics_dynamic * physics_loss

    parts = {
        "zone": float(zone_loss.detach().cpu()),
        "offset": float(offset_loss.detach().cpu()),
        "step": float(step_loss.detach().cpu()),
        "physics": float(physics_loss.detach().cpu()),
        "data": float(data_loss.detach().cpu()),
        "w_physics_dynamic": float(w_physics_dynamic.detach().cpu()),
    }
    return total, parts

# =========================================================


## Step 10: Feature Engineering, Normalization, Split, and Training Utilities

This step packs row dictionaries into arrays, computes transient-style convergence diagnostics, standardizes features using training statistics, performs the original train/validation split, trains the model, and defines single-curve inference.

**Why this matters for partial-shading MPPT:** normalization and a held-out validation set let the network learn from sparse scans while still providing an independent check during training.

**What to check:** training logs should include train loss, validation loss, zone accuracy, voltage percentage error, dynamic physics weight, and convergence-step estimates.


In [ ]:
# TRAIN / PREDICT
# =========================================================
def rows_to_arrays(rows: List[Dict[str, Any]]) -> Dict[str, np.ndarray]:
    """Pack per-curve feature dictionaries into NumPy arrays for splitting/training."""
    """Convert feature dictionaries into aligned NumPy arrays for model training/eval."""
    seq_vhat = np.stack([r["sparse_vhat"] for r in rows], axis=0).astype(np.float32)
    seq_ihat = np.stack([r["sparse_ihat"] for r in rows], axis=0).astype(np.float32)
    out = {
        "x": np.stack([r["x"] for r in rows], axis=0).astype(np.float32),
        "y_zone": np.asarray([r["y_zone"] for r in rows], dtype=np.int64),
        "y_offset": np.asarray([r["y_offset"] for r in rows], dtype=np.float32),
        "y_step_ratio": np.asarray([r["y_step_ratio"] for r in rows], dtype=np.float32),
        "true_vhat": np.asarray([r["true_vhat"] for r in rows], dtype=np.float32),
        "seq_vhat": seq_vhat,
        "seq_ihat": seq_ihat,
    }
    return out


def compute_transient_convergence_steps(
    pred_vhat: torch.Tensor,
    step_ratio_pred: torch.Tensor,
    true_vhat: torch.Tensor,
    cfg: Config,
    tol_ratio: float = 0.01,
    stable_window: int = 3,
    max_steps: int = 64,
) -> float:
    pred = torch.clamp(pred_vhat.detach(), cfg.sample_fracs_min, cfg.sample_fracs_max)
    truth = torch.clamp(true_vhat.detach(), cfg.sample_fracs_min, cfg.sample_fracs_max)
    step = torch.clamp(step_ratio_pred.detach(), cfg.po_min_step_ratio, cfg.po_max_step_ratio)
    max_steps = max(max_steps, stable_window + 1)

    convergence_steps: List[float] = []
    for p0, s, t in zip(pred, step, truth):
        v_now = float(p0.item())
        step_now = float(s.item())
        target = float(t.item())
        tol = max(tol_ratio * abs(target), 1e-6)
        stable_count = 0
        step_hit = float(max_steps)
        for k in range(1, max_steps + 1):
            direction = 1.0 if target > v_now else -1.0
            v_now = float(np.clip(v_now + direction * step_now, cfg.sample_fracs_min, cfg.sample_fracs_max))
            if abs(v_now - target) <= tol:
                stable_count += 1
            else:
                stable_count = 0
            if stable_count >= stable_window:
                step_hit = float(k - stable_window + 1)
                break
        convergence_steps.append(step_hit)
    return float(np.mean(convergence_steps)) if convergence_steps else float(max_steps)


def fit_standardizer(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Fit mean/std on training features only to prevent validation leakage."""
    mu = X.mean(axis=0, keepdims=True).astype(np.float32)
    sd = (X.std(axis=0, keepdims=True) + 1e-8).astype(np.float32)
    return mu, sd


def apply_standardizer(X: np.ndarray, mu: np.ndarray, sd: np.ndarray) -> np.ndarray:
    """Apply precomputed training-set standardization parameters."""
    return ((X - mu) / sd).astype(np.float32)


def train_model(sim_rows: List[Dict[str, Any]], cfg: Config):
    """Train Physics-Informed TCNformer with holdout validation and early stopping."""
    # Preserve the original random holdout split behavior for validation.
    idx = np.arange(len(sim_rows))
    idx_tr, idx_va = train_test_split(idx, test_size=cfg.sim_holdout_fraction, random_state=cfg.seed, shuffle=True)
    train_rows = [sim_rows[i] for i in idx_tr]
    val_rows = [sim_rows[i] for i in idx_va]

    tr = rows_to_arrays(train_rows)
    va = rows_to_arrays(val_rows)
    mu, sd = fit_standardizer(tr["x"])
    tr_x = apply_standardizer(tr["x"], mu, sd)
    va_x = apply_standardizer(va["x"], mu, sd)

    model = PhysicsInformedTCNformer(in_dim=tr_x.shape[1], n_zones=cfg.n_zones, cfg=cfg).to(cfg.device)
    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)
    zone_edges_t = torch.tensor(cfg.zone_edges, dtype=torch.float32, device=cfg.device)

    best_val = float('inf')
    best_state = None
    patience = cfg.early_stop_patience
    history: List[Dict[str, float]] = []

    for ep in range(1, cfg.epochs + 1):
        model.train()
        perm = np.random.permutation(len(tr_x))
        train_loss_sum = 0.0
        n_seen = 0
        train_parts_acc = {k: 0.0 for k in ["zone", "offset", "step", "physics", "data", "w_physics_dynamic"]}
        for st in range(0, len(perm), cfg.batch_size):
            bi = perm[st:st + cfg.batch_size]
            if len(bi) < 2:
                continue
            batch = {
                "x_std": torch.tensor(tr_x[bi], dtype=torch.float32, device=cfg.device),
                "y_zone": torch.tensor(tr["y_zone"][bi], dtype=torch.long, device=cfg.device),
                "y_offset": torch.tensor(tr["y_offset"][bi], dtype=torch.float32, device=cfg.device),
                "y_step_ratio": torch.tensor(tr["y_step_ratio"][bi], dtype=torch.float32, device=cfg.device),
                "true_vhat": torch.tensor(tr["true_vhat"][bi], dtype=torch.float32, device=cfg.device),
                "seq_vhat": torch.tensor(tr["seq_vhat"][bi], dtype=torch.float32, device=cfg.device),
                "seq_ihat": torch.tensor(tr["seq_ihat"][bi], dtype=torch.float32, device=cfg.device),
            }
            # Loss combines zone classification, offset/step regression, and the physics term.
            optimizer.zero_grad()
            zone_probs, offset_pred, step_ratio_pred = model(batch["x_std"])
            loss, parts = pinn_zone_offset_loss(zone_probs, offset_pred, step_ratio_pred, batch, cfg, zone_edges_t)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            optimizer.step()
            bs = len(bi)
            train_loss_sum += float(loss.detach().cpu()) * bs
            n_seen += bs
            for k in train_parts_acc:
                train_parts_acc[k] += parts[k] * bs

        avg_train_loss = train_loss_sum / max(n_seen, 1)
        for k in train_parts_acc:
            train_parts_acc[k] /= max(n_seen, 1)

        model.eval()
        vb = {
            "x_std": torch.tensor(va_x, dtype=torch.float32, device=cfg.device),
            "y_zone": torch.tensor(va["y_zone"], dtype=torch.long, device=cfg.device),
            "y_offset": torch.tensor(va["y_offset"], dtype=torch.float32, device=cfg.device),
            "y_step_ratio": torch.tensor(va["y_step_ratio"], dtype=torch.float32, device=cfg.device),
            "true_vhat": torch.tensor(va["true_vhat"], dtype=torch.float32, device=cfg.device),
            "seq_vhat": torch.tensor(va["seq_vhat"], dtype=torch.float32, device=cfg.device),
            "seq_ihat": torch.tensor(va["seq_ihat"], dtype=torch.float32, device=cfg.device),
        }
        # Validation metrics are computed on the held-out split without changing model weights.
        zone_probs, offset_pred, step_ratio_pred = model(vb["x_std"])
        val_loss, val_parts = pinn_zone_offset_loss(zone_probs, offset_pred, step_ratio_pred, vb, cfg, zone_edges_t)
        scheduler.step(ep)
        pred_vhat, zone_probs, _, _ = compute_pred_vhat(zone_probs, offset_pred, zone_edges_t)
        zone_acc = float((torch.argmax(zone_probs, dim=1) == vb["y_zone"]).float().mean().detach().cpu() * 100.0)
        convergence_steps = compute_transient_convergence_steps(pred_vhat, step_ratio_pred, vb["true_vhat"], cfg)
        val_v_pct = float((torch.abs(pred_vhat - (0.5*(zone_edges_t[vb['y_zone']] + zone_edges_t[vb['y_zone']+1]) + 0.5*(zone_edges_t[vb['y_zone']+1]-zone_edges_t[vb['y_zone']])*vb['y_offset'])) / torch.clamp(torch.abs(0.5*(zone_edges_t[vb['y_zone']] + zone_edges_t[vb['y_zone']+1]) + 0.5*(zone_edges_t[vb['y_zone']+1]-zone_edges_t[vb['y_zone']])*vb['y_offset']), min=1e-6)).mean().detach().cpu() * 100.0)
        val_loss_f = float(val_loss.detach().cpu())

        history.append({
            "epoch": ep,
            "train_loss": avg_train_loss,
            "val_loss": val_loss_f,
            "zone_acc": zone_acc,
            "val_v_pct": val_v_pct,
            "convergence_steps": convergence_steps,
            **{f"train_{k}": v for k, v in train_parts_acc.items()},
            **{f"val_{k}": v for k, v in val_parts.items()},
        })
        print(
            f"Epoch {ep:03d} | train={avg_train_loss:.6f} | val={val_loss_f:.6f} | zone_acc={zone_acc:.2f}% | "
            f"val_v_pct={val_v_pct:.3f}% | val_data={val_parts['data']:.4f} | val_physics={val_parts['physics']:.4f} | "
            f"w_phys_dyn={val_parts['w_physics_dynamic']:.4f} | convergence_steps={convergence_steps:.2f}"
        )
        if val_loss_f < best_val - cfg.early_stop_min_delta:
            best_val = val_loss_f
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience = cfg.early_stop_patience
        else:
            patience -= 1
            if patience <= 0:
                print("Early stopping triggered.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, mu, sd, history, train_rows, val_rows


def predict_for_oracle(oracle: CurveOracle, model, mu: np.ndarray, sd: np.ndarray, cfg: Config) -> Dict[str, Any]:
    """Run one cleaned PV curve through feature creation, standardization, and inference."""
    feat = build_feature_vector(*oracle.curve_for_plot(), cfg.sample_fracs, cfg)
    x_std = apply_standardizer(feat["x"][None, :], mu, sd)
    x_t = torch.tensor(x_std, dtype=torch.float32, device=cfg.device)
    zone_edges_t = torch.tensor(cfg.zone_edges, dtype=torch.float32, device=cfg.device)
    model.eval()
    with torch.no_grad():
        zone_probs, offset_pred, step_ratio = model(x_t)
        pred_vhat, zone_probs, _, zone_conf = compute_pred_vhat(zone_probs, offset_pred, zone_edges_t)
    pred_vhat_f = float(pred_vhat.detach().cpu().numpy().ravel()[0])
    V_pred = float(np.clip(pred_vhat_f * oracle.voc, cfg.sample_fracs_min * oracle.voc, cfg.sample_fracs_max * oracle.voc))
    P_pred = float(oracle.power_at(V_pred))
    return {
        "feature": feat,
        "zone_conf": float(zone_conf.detach().cpu().numpy().ravel()[0]),
        "pred_vhat": pred_vhat_f,
        "V_pred": V_pred,
        "P_pred": P_pred,
        "step_ratio": float(step_ratio.detach().cpu().numpy().ravel()[0]),
    }

# =========================================================
# LOCAL P&O (adjusted step size, ANN seed only)
# =========================================================


## Step 11: Local P&O Refinement

This step defines the local Perturb-and-Observe refinement and the unchanged coarse 12-point baseline. The neural prediction provides a seed voltage and step ratio, then P&O locally probes power around that seed.

**Why this matters for partial-shading MPPT:** the neural seed helps avoid starting in the wrong region, while local P&O can recover additional power by following the nearby peak.

**What to check:** final evaluation should report refined voltage/power and the P&O trace length for representative examples.


In [ ]:
def local_po_adjusted(oracle: CurveOracle, v_seed: float, step_ratio: float, cfg: Config) -> Tuple[float, float, List[Tuple[float, float]]]:
    """Refine the neural seed with the original local Perturb-and-Observe routine.

    The neural network supplies a starting voltage and step ratio; P&O then searches
    locally on the same oracle to improve final tracking power under partial shading.
    """
    voc = max(oracle.voc, 1e-9)
    step = float(np.clip(step_ratio, cfg.po_min_step_ratio, cfg.po_max_step_ratio)) * voc
    v_now = float(np.clip(v_seed, cfg.sample_fracs_min * voc, cfg.sample_fracs_max * voc))
    p_now = oracle.power_at(v_now)
    trace = [(v_now, p_now)]

    # Initial left/right probing decides the first local hill-climb direction.
    v_left = float(np.clip(v_now - step, cfg.sample_fracs_min * voc, cfg.sample_fracs_max * voc))
    v_right = float(np.clip(v_now + step, cfg.sample_fracs_min * voc, cfg.sample_fracs_max * voc))
    p_left = oracle.power_at(v_left)
    p_right = oracle.power_at(v_right)
    trace.extend([(v_left, p_left), (v_right, p_right)])

    if p_left > p_now and p_left >= p_right:
        direction = -1.0
        v_now, p_now = v_left, p_left
    elif p_right > p_now:
        direction = 1.0
        v_now, p_now = v_right, p_right
    else:
        direction = 1.0 if p_right >= p_left else -1.0
        step *= cfg.po_step_shrink

    for _ in range(cfg.po_iterations):
        v_try = float(np.clip(v_now + direction * step, cfg.sample_fracs_min * voc, cfg.sample_fracs_max * voc))
        p_try = oracle.power_at(v_try)
        trace.append((v_try, p_try))
        gain_ratio = (p_try - p_now) / max(abs(p_now), 1e-9)
        if p_try > p_now:
            v_now, p_now = v_try, p_try
            step = min(step * cfg.po_step_growth, cfg.po_max_step_ratio * voc)
            if gain_ratio < cfg.po_stop_gain_ratio:
                break
        else:
            direction *= -1.0
            step = max(step * cfg.po_step_shrink, cfg.po_min_step_ratio * voc)
            if step <= cfg.po_min_step_ratio * voc + 1e-12:
                break
    return float(v_now), float(p_now), trace


def coarse12_baseline(oracle: CurveOracle, cfg: Config) -> Tuple[float, float, np.ndarray, np.ndarray]:
    """Return the best point from the unchanged 12-sample sparse scan baseline."""
    vq = (cfg.sample_fracs * oracle.voc).astype(float)
    pq = np.array([oracle.power_at(v) for v in vq], dtype=float)
    k = int(np.argmax(pq))
    return float(vq[k]), float(pq[k]), vq, pq

# =========================================================


## Step 12: Evaluation Metrics

This step evaluates both the seed-only TCNformer prediction and the final TCNformer+P&O hybrid result. Metrics include power ratio, power regret, voltage difference, zone confidence, dense peak count, and comparison with the coarse 12-point baseline.

**Why this matters for partial-shading MPPT:** good MPPT performance means high recovered power and low regret relative to the dense true MPP, especially when multiple peaks exist.

**What to check:** the final mean/median power ratios should be high, regret should be low, and the refined method should be interpreted separately from the seed-only prediction.


In [ ]:
# EVALUATION
# =========================================================
def evaluate_tcnformer_only(curves, model, mu, sd, cfg: Config):
    """Evaluate seed-only neural predictions against dense true MPP references."""
    rows: List[Dict[str, Any]] = []
    for idx, item in enumerate(curves[: cfg.max_eval_curves]):
        oracle = CurveOracle(item)
        if oracle.voc <= 0 or oracle.pmpp_true <= 0:
            continue
        pred = predict_for_oracle(oracle, model, mu, sd, cfg)
        rows.append({
            "curve_idx": idx,
            "Voc": float(oracle.voc),
            "V_true": float(oracle.vmpp_true),
            "P_true": float(oracle.pmpp_true),
            "V_tcnformer": pred["V_pred"],
            "P_tcnformer": pred["P_pred"],
            "tcnformer_ratio_pct": 100.0 * pred["P_pred"] / max(oracle.pmpp_true, 1e-9),
            "tcnformer_power_regret_pct": 100.0 * max(oracle.pmpp_true - pred["P_pred"], 0.0) / max(oracle.pmpp_true, 1e-9),
            "tcnformer_vdiff_pct": 100.0 * abs(pred["V_pred"] - oracle.vmpp_true) / max(abs(oracle.vmpp_true), 1e-9),
            "zone_conf": pred["zone_conf"],
            "tcnformer_step_ratio": pred["step_ratio"],
            "dense_peak_count": int(oracle.dense_peak_count),
        })
    df = pd.DataFrame(rows)
    if len(df) == 0:
        return {}, df
    summary = {
        "mean_tcnformer_ratio_pct": float(df["tcnformer_ratio_pct"].mean()),
        "median_tcnformer_ratio_pct": float(df["tcnformer_ratio_pct"].median()),
        "mean_tcnformer_power_regret_pct": float(df["tcnformer_power_regret_pct"].mean()),
        "median_tcnformer_power_regret_pct": float(df["tcnformer_power_regret_pct"].median()),
        "max_tcnformer_power_regret_pct": float(df["tcnformer_power_regret_pct"].max()),
        "mean_tcnformer_vdiff_pct": float(df["tcnformer_vdiff_pct"].mean()),
        "median_tcnformer_vdiff_pct": float(df["tcnformer_vdiff_pct"].median()),
        "tcnformer_within_99pct_power_rate": float(100.0 * (df["tcnformer_ratio_pct"] >= 99.0).mean()),
        "tcnformer_within_98pct_power_rate": float(100.0 * (df["tcnformer_ratio_pct"] >= 98.0).mean()),
        "tcnformer_within_95pct_power_rate": float(100.0 * (df["tcnformer_ratio_pct"] >= 95.0).mean()),
        "mean_zone_conf": float(df["zone_conf"].mean()),
        "mean_tcnformer_step_ratio": float(df["tcnformer_step_ratio"].mean()),
    }
    return summary, df


def evaluate_tcnformer_plus_po(curves, model, mu, sd, cfg: Config):
    """Evaluate final hybrid tracking after neural seeding plus local P&O refinement."""
    rows: List[Dict[str, Any]] = []
    examples: List[Dict[str, Any]] = []
    for idx, item in enumerate(curves[: cfg.max_eval_curves]):
        oracle = CurveOracle(item)
        if oracle.voc <= 0 or oracle.pmpp_true <= 0:
            continue
        pred = predict_for_oracle(oracle, model, mu, sd, cfg)
        V_samples = (cfg.sample_fracs * oracle.voc).astype(float)
        P_samples = np.array([oracle.power_at(vq) for vq in V_samples], dtype=float)
        V_final, P_final, po_trace = local_po_adjusted(oracle, pred["V_pred"], pred["step_ratio"], cfg)
        row = {
            "curve_idx": idx,
            "Voc": float(oracle.voc),
            "V_true": float(oracle.vmpp_true),
            "P_true": float(oracle.pmpp_true),
            "V_tcnformer": pred["V_pred"],
            "P_tcnformer": pred["P_pred"],
            "V_final": V_final,
            "P_final": P_final,
            "tcnformer_ratio_pct": 100.0 * pred["P_pred"] / max(oracle.pmpp_true, 1e-9),
            "tcnformer_power_regret_pct": 100.0 * max(oracle.pmpp_true - pred["P_pred"], 0.0) / max(oracle.pmpp_true, 1e-9),
            "tcnformer_vdiff_pct": 100.0 * abs(pred["V_pred"] - oracle.vmpp_true) / max(abs(oracle.vmpp_true), 1e-9),
            "final_ratio_pct": 100.0 * P_final / max(oracle.pmpp_true, 1e-9),
            "final_power_regret_pct": 100.0 * max(oracle.pmpp_true - P_final, 0.0) / max(oracle.pmpp_true, 1e-9),
            "final_vdiff_pct": 100.0 * abs(V_final - oracle.vmpp_true) / max(abs(oracle.vmpp_true), 1e-9),
            "zone_conf": pred["zone_conf"],
            "tcnformer_step_ratio": pred["step_ratio"],
            "dense_peak_count": int(oracle.dense_peak_count),
        }
        rows.append(row)
        examples.append({
            **row,
            "oracle": oracle,
            "V_samples": V_samples,
            "P_samples": P_samples,
            "po_trace": po_trace,
            "plot_peak_count": int(oracle.dense_peak_count),
            "peak_prominence_score": float(oracle.peak_prominence_score),
            "v_dense": oracle.v_dense,
            "p_dense": oracle.p_dense,
        })
    df = pd.DataFrame(rows)
    if len(df) == 0:
        return {}, df, examples
    summary = {
        "mean_tcnformer_ratio_pct": float(df["tcnformer_ratio_pct"].mean()),
        "median_tcnformer_ratio_pct": float(df["tcnformer_ratio_pct"].median()),
        "mean_tcnformer_power_regret_pct": float(df["tcnformer_power_regret_pct"].mean()),
        "mean_tcnformer_vdiff_pct": float(df["tcnformer_vdiff_pct"].mean()),
        "tcnformer_within_99pct_power_rate": float(100.0 * (df["tcnformer_ratio_pct"] >= 99.0).mean()),
        "tcnformer_within_98pct_power_rate": float(100.0 * (df["tcnformer_ratio_pct"] >= 98.0).mean()),
        "tcnformer_within_95pct_power_rate": float(100.0 * (df["tcnformer_ratio_pct"] >= 95.0).mean()),
        "mean_final_ratio_pct": float(df["final_ratio_pct"].mean()),
        "median_final_ratio_pct": float(df["final_ratio_pct"].median()),
        "mean_final_power_regret_pct": float(df["final_power_regret_pct"].mean()),
        "median_final_power_regret_pct": float(df["final_power_regret_pct"].median()),
        "max_final_power_regret_pct": float(df["final_power_regret_pct"].max()),
        "mean_final_vdiff_pct": float(df["final_vdiff_pct"].mean()),
        "final_within_99pct_power_rate": float(100.0 * (df["final_ratio_pct"] >= 99.0).mean()),
        "final_within_98pct_power_rate": float(100.0 * (df["final_ratio_pct"] >= 98.0).mean()),
        "final_within_95pct_power_rate": float(100.0 * (df["final_ratio_pct"] >= 95.0).mean()),
        "mean_zone_conf": float(df["zone_conf"].mean()),
        "mean_tcnformer_step_ratio": float(df["tcnformer_step_ratio"].mean()),
    }
    return summary, df, examples

# =========================================================


## Step 13: Visualization and Final Results

This step plots training history and selected shaded examples. The plots compare the dense P-V curve, true MPP, neural seed, refined P&O result, and coarse 12-point sample baseline.

**Why this matters for partial-shading MPPT:** visual inspection shows whether the hybrid tracker chooses the global region instead of a lower local peak.

**What to check:** on each plot, compare marker positions and power levels; the refined point should ideally be close to the true MPP and above weaker local maxima.


In [ ]:
# VISUALIZATION
# =========================================================

# =========================================================
# VISUALIZATION
# =========================================================
def select_visualization_examples(examples: List[Dict[str, Any]], n_viz: int = 8):
    if len(examples) == 0:
        return [], 'none'
    true_multi = [e for e in examples if int(e.get('plot_peak_count', 0)) >= 2]
    if len(true_multi) > 0:
        chosen = sorted(true_multi, key=lambda x: (x.get('peak_prominence_score', 0.0), x.get('final_ratio_pct', 0.0)), reverse=True)[:n_viz]
        return chosen, 'true_two_peak'
    chosen = sorted(examples, key=lambda x: (x.get('peak_prominence_score', 0.0), x.get('final_ratio_pct', 0.0)), reverse=True)[:n_viz]
    return chosen, 'highest_prominence_fallback'


def plot_training_histories(history: List[Dict[str, float]]):
    """Plot train/validation loss so reviewers can inspect optimization behavior."""
    df = pd.DataFrame(history)
    if len(df) == 0:
        return
    plt.figure(figsize=(10, 5))
    plt.plot(df['epoch'], df['train_loss'], label='train_loss')
    plt.plot(df['epoch'], df['val_loss'], label='val_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


def visualize_examples_from_full_shaded_pool(examples: List[Dict[str, Any]], n_viz: int = 8):
    """Plot true, neural-seed, refined, and coarse-scan points on representative curves."""
    title_font_size = 13
    axis_label_font_size = 14
    tick_font_size = 12
    legend_font_size = 12
    chosen, mode = select_visualization_examples(examples, n_viz=n_viz)
    if len(chosen) == 0:
        print('No valid shaded-experimental visualization examples were found.')
        return
    if mode == 'true_two_peak':
        print('Visualization mode: strongest true two-peak curves from the full shaded-experimental pool')
    else:
        print('Visualization mode: no curves passed the strict two-peak threshold; showing highest-prominence shaded curves instead')
    n_rows = int(np.ceil(len(chosen) / 2))
    fig, axes = plt.subplots(n_rows, 2, figsize=(15, 4 * n_rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[len(chosen):]:
        ax.axis('off')
    for ax, item in zip(axes[:len(chosen)], chosen):
        oracle = item['oracle']
        v_plot = item.get('v_dense', None)
        p_plot = item.get('p_dense', None)
        if v_plot is None or len(v_plot) == 0:
            v, i = oracle.curve_for_plot()
            v_plot, p_plot = prepare_pv_plot_curve(v, i)
        ax.plot(v_plot, p_plot, label='Plot-ready P-V curve')
        ax.scatter(item['V_samples'], item['P_samples'], s=20, alpha=0.65, label='12 sample points')
        ax.scatter([oracle.vmpp_true], [oracle.pmpp_true], marker='*', s=160, label='True MPP', zorder=5)
        ax.scatter([item['V_tcnformer']], [item['P_tcnformer']], marker='x', s=90, label='TCNformer stage', zorder=6)
        ax.scatter([item['V_final']], [item['P_final']], marker='D', s=70, label='TCNformer + local P&O', zorder=6)
        if item.get('po_trace'):
            po_v = [vp[0] for vp in item['po_trace']]
            po_p = [vp[1] for vp in item['po_trace']]
            ax.plot(po_v, po_p, linestyle='--', alpha=0.6, label='Local P&O trace')
        ax.set_title(
            f"Curve {item['curve_idx']} | peaks={item.get('plot_peak_count', 0)} | "
            f"TCNformer={item['tcnformer_ratio_pct']:.2f}% | Final={item['final_ratio_pct']:.2f}%",
            fontsize=title_font_size,
        )
        ax.set_xlabel('Voltage (V)', fontsize=axis_label_font_size)
        ax.set_ylabel('Power (W)', fontsize=axis_label_font_size)
        ax.tick_params(axis='both', labelsize=tick_font_size)
        ax.grid(True, alpha=0.3)
    axes[0].legend(loc='best', fontsize=legend_font_size)
    plt.tight_layout()
    plt.show()

# =========================================================


## Step 14: Main Execution and Final Verification

This step runs the full workflow in order: load data, build cleaned rows, train the model, evaluate shaded curves, optionally evaluate normal experimental curves, plot results, and save the model bundle.

**Why this matters for partial-shading MPPT:** this is the end-to-end verification that the documented notebook still follows the same hybrid MPPT process after the readability cleanup.

**What to check:** review preprocessing summaries, seed-only and refined evaluation summaries, plots, and any saved bundle message. If the dataset is missing, this section will stop at validation just as the original notebook would.


In [ ]:
# RUN PIPELINE
# Execution flow:
# 1) load dataset, 2) preprocess/clean curves, 3) print preprocessing stats,
# 4) train TCNformer, 5) evaluate shaded set, 6) optional normal-set evaluation,
# 7) visualize histories/examples, 8) save bundle.
# =========================================================
sim_curves, exp_sh_curves, exp_ok_curves = load_dataset(DATASET_PATH)

# Standardize uploaded data using one dataset-level format before
# build_dataset_from_curves so the legacy downstream P = V * I calculation
# receives pseudo-current for PV_Data_Reduced.mat direct P-V curves.
print(f"\nStandardizing curves with dataset format: {NEW_DATASET_FORMAT}")
sim_curves = standardize_curves_by_format(sim_curves, NEW_DATASET_FORMAT)
exp_sh_curves = standardize_curves_by_format(exp_sh_curves, NEW_DATASET_FORMAT)
exp_ok_curves = standardize_curves_by_format(exp_ok_curves, NEW_DATASET_FORMAT) if len(exp_ok_curves) > 0 else []

print("\nBuilding cleaned training dataset from simulated curves...")
sim_rows, sim_stats = build_dataset_from_curves(sim_curves, cfg)
exp_sh_rows, exp_sh_stats = build_dataset_from_curves(exp_sh_curves, cfg)
exp_ok_rows, exp_ok_stats = build_dataset_from_curves(exp_ok_curves, cfg) if len(exp_ok_curves) > 0 else ([], {})

print("\n=== CLEANING / PREPROCESSING SUMMARY ===")
print("Simulated stats:", sim_stats)
print("Shaded experimental stats:", exp_sh_stats)
if REPORT_OK_EXPERIMENTAL_AUX:
    print("Ok experimental stats:", exp_ok_stats)

if len(sim_rows) == 0:
    raise RuntimeError("No valid simulated training samples after preprocessing.")
if len(exp_sh_rows) == 0:
    raise RuntimeError("No valid shaded experimental samples after preprocessing.")

print("Feature matrix shape:", rows_to_arrays(sim_rows)["x"].shape)
print("Target type: 12-point samples -> ANN (zone+offset+step) -> local adjusted-step P&O")
print("Input dimension:", rows_to_arrays(sim_rows)["x"].shape[1])
print("Number of zones:", cfg.n_zones)
print("Zone edges:", cfg.zone_edges)

print("\nTraining PINN-guided Zone-and-Offset ANN...")
model, mu, sd, history, sim_train_rows, sim_holdout_rows = train_model(sim_rows, cfg)

print("\nQuantitative evaluation source: full unseen shaded-experimental set")
tcnformer_summary_exp_sh, tcnformer_df_exp_sh = evaluate_tcnformer_only([r['curve'] for r in exp_sh_rows], model, mu, sd, cfg)
hybrid_summary_exp_sh, hybrid_df_exp_sh, exp_sh_examples = evaluate_tcnformer_plus_po([r['curve'] for r in exp_sh_rows], model, mu, sd, cfg)

tcnformer_summary_sim_hold, tcnformer_df_sim_hold = evaluate_tcnformer_only([r['curve'] for r in sim_holdout_rows], model, mu, sd, cfg)

print("\n=== SIM HOLDOUT TCNformer-ONLY SUMMARY ===")
for k, v in tcnformer_summary_sim_hold.items():
    print(f"{k}: {v}")

print("\n=== EXP SHADED TCNformer-ONLY SUMMARY ===")
for k, v in tcnformer_summary_exp_sh.items():
    print(f"{k}: {v}")

print("\n=== EXP SHADED TCNformer + LOCAL P&O SUMMARY ===")
for k, v in hybrid_summary_exp_sh.items():
    print(f"{k}: {v}")

print("\nUnseen shaded experimental TCNformer-only head:")
display(tcnformer_df_exp_sh.head(10))
print("\nUnseen shaded experimental TCNformer + local P&O head:")
display(hybrid_df_exp_sh.head(10))

if REPORT_OK_EXPERIMENTAL_AUX and len(exp_ok_rows) > 0:
    print("\n=== AUXILIARY OK EXPERIMENTAL TCNformer + LOCAL P&O SUMMARY ===")
    ok_summary, ok_df, _ = evaluate_tcnformer_plus_po([r['curve'] for r in exp_ok_rows], model, mu, sd, cfg)
    for k, v in ok_summary.items():
        print(f"{k}: {v}")
    display(ok_df.head(10))

if MAKE_PLOTS:
    plot_training_histories(history)
    visualize_examples_from_full_shaded_pool(exp_sh_examples, n_viz=cfg.n_viz)

if SAVE_MODEL_BUNDLE:
    bundle = {
        "config": cfg.__dict__,
        "mu": mu,
        "sd": sd,
        "history": history,
        "tcnformer_summary_exp_sh": tcnformer_summary_exp_sh,
        "hybrid_summary_exp_sh": hybrid_summary_exp_sh,
        "state_dict": model.state_dict(),
    }
    save_path = "zone_offset_ann_pinn_12sample_ann_po_bundle.pt"
    torch.save(bundle, save_path)
    print("\nSaved model bundle to:", save_path)

## Step 15: Inference Notes and Reviewer Checklist

For inference on one curve, the same steps are used: clean the curve, create the 12-point sparse features, standardize with training statistics, predict a zone/offset seed, and apply local P&O refinement.

**Why this matters for partial-shading MPPT:** applying the exact same preprocessing and feature normalization at inference prevents train/deployment mismatch.

**What to check:** any reported final MPP should identify whether it is the neural seed, the locally refined output, or the dense true reference used only for evaluation.
